# DCAMF-Net 模型内部可视化

跟着一段水声信号，逐步看网络每一层做了什么。

## 1. 加载模型 + 测试样本

In [ ]:
import os, sys, json
import torch, numpy as np, soundfile as sf
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import welch, spectrogram

os.environ['OMP_NUM_THREADS'] = '8'
PROJECT_ROOT = Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'dcamf_net'))

from dcamf_net.model import DCAMFNet
from argparse import Namespace

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

args = Namespace(enc_channels=256, enc_kernel_size=80, enc_stride=40,
                 chunk_size=500, hop_size=250, n_blocks=10, n_heads=8,
                 ffn_hidden=512, dw_kernel_size=31, checkpoint=None)

model = DCAMFNet(in_channels=1, enc_channels=256, enc_kernel_size=80, enc_stride=40,
                 chunk_size=500, hop_size=250, n_blocks=10, n_heads=8,
                 ffn_hidden=512, dw_kernel_size=31).to(device)

ckpt = torch.load(PROJECT_ROOT / 'experiments' / 'dcamf_net' / 'checkpoints' / 'best_SISNR.pth',
                  map_location=device, weights_only=False)
sd = {k: v for k, v in ckpt.items()
      if not k.endswith('.total_ops') and not k.endswith('.total_params')
      and k not in ('total_ops', 'total_params')}
model.load_state_dict(sd)
model.eval()
print(f'Model loaded. Params: {sum(p.numel() for p in model.parameters()):,}')

clean, sr = sf.read(str(PROJECT_ROOT / 'data' / 'ShipsEar' / 'test1' / 'clean' /
    sorted(list((PROJECT_ROOT / 'data' / 'ShipsEar' / 'test1' / 'clean').glob('*.wav')))[0].name))
noisy, _ = sf.read(str(PROJECT_ROOT / 'data' / 'ShipsEar' / 'test1' / 'noisy' /
    sorted(list((PROJECT_ROOT / 'data' / 'ShipsEar' / 'test1' / 'noisy').glob('*.wav')))[0].name))

if clean.ndim > 1: clean = clean.mean(axis=1)
if noisy.ndim > 1: noisy = noisy.mean(axis=1)
L = 48000
clean, noisy = clean[:L], noisy[:L]
noisy_t = torch.from_numpy(noisy).float().unsqueeze(0).unsqueeze(0).to(device)
print(f'Sample: {L} samples, {L/sr:.1f}s, SR={sr}Hz')


## 2. 设置钩子 - 抓取每个模块的输出

In [ ]:
activations = {}

def hook_fn(name):
    def hook(module, input, output):
        if isinstance(output, torch.Tensor):
            activations[name] = output.detach().cpu()
        elif isinstance(output, (list, tuple)):
            activations[name] = [o.detach().cpu() if isinstance(o, torch.Tensor) else o for o in output]
    return hook

hooks = []
hooks.append(model.encoder.conv.register_forward_hook(hook_fn('encoder_conv_out')))
hooks.append(model.encoder.prelu.register_forward_hook(hook_fn('encoder_prelu_out')))
hooks.append(model.encoder.register_forward_hook(hook_fn('encoder_full_out')))

b0 = model.dcam_blocks[0]
hooks.append(b0.global_cemhsa.register_forward_hook(hook_fn('block0_global_cemhsa')))
hooks.append(b0.local_cemhsa.register_forward_hook(hook_fn('block0_local_cemhsa')))
hooks.append(b0.register_forward_hook(hook_fn('block0_full_out')))

b9 = model.dcam_blocks[9]
hooks.append(b9.global_cemhsa.register_forward_hook(hook_fn('block9_global_cemhsa')))
hooks.append(b9.local_cemhsa.register_forward_hook(hook_fn('block9_local_cemhsa')))
hooks.append(b9.register_forward_hook(hook_fn('block9_full_out')))

hooks.append(model.decoder.register_forward_hook(hook_fn('decoder_out')))
print(f'Registered {len(hooks)} hooks')


## 3. 前向传播

In [ ]:
with torch.no_grad():
    enhanced = model(noisy_t)

enhanced_np = enhanced.squeeze().cpu().numpy()
print(f'Input shape:  {noisy_t.shape}')
print(f'Output shape: {enhanced.shape}')
print('Captured intermediate activations:')
for name, val in activations.items():
    if isinstance(val, torch.Tensor):
        print(f'  {name:30s} shape={list(val.shape)}')


## 4. 输入输出对比

In [ ]:
import IPython.display as ipd

fig, axes = plt.subplots(3, 1, figsize=(14, 7))
t = np.arange(L) / sr
for ax, sig, title, c in zip(axes,
    [clean, noisy, enhanced_np],
    ['Clean (Target)', 'Noisy (Input)', 'DCAMF-Net (Output)'],
    ['black', '#E74C3C', '#1C7293']):
    ax.plot(t, sig, color=c, linewidth=0.5)
    ax.set_title(title); ax.set_xlim(0, 0.3)
plt.tight_layout(); plt.show()

print('Noisy:'); ipd.display(ipd.Audio(noisy, rate=sr))
print('Enhanced:'); ipd.display(ipd.Audio(enhanced_np, rate=sr))
print('Clean:'); ipd.display(ipd.Audio(clean, rate=sr))


## 5. 编码器 - 256个分析滤波器分解信号

In [ ]:
enc_out = activations['encoder_conv_out']
print(f'Encoder Conv1d output shape: {list(enc_out.shape)}')
print(f'256 channels = 256 learned filters, each ~5ms wide')

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
t_wav = np.arange(L) / sr * 1000
axes[0].plot(t_wav, noisy, color='#E74C3C', linewidth=0.4)
axes[0].set_title('Input Waveform (Noisy)'); axes[0].set_xlabel('Time (ms)')
axes[0].set_xlim(0, 500)

feat = enc_out[0, :16].numpy()
t_feat = np.arange(feat.shape[1]) * 40 / sr * 1000
im = axes[1].imshow(feat, aspect='auto', origin='lower', cmap='RdBu_r',
                     extent=[t_feat[0], t_feat[-1], 0, 16])
axes[1].set_title('Encoder: First 16 Filter Outputs (256 total)')
axes[1].set_xlabel('Time (ms)'); axes[1].set_ylabel('Filter #')
plt.colorbar(im, ax=axes[1])

enc_active = activations['encoder_prelu_out'][0, :16].numpy()
axes[2].imshow(enc_active, aspect='auto', origin='lower', cmap='RdBu_r',
                extent=[t_feat[0], t_feat[-1], 0, 16])
axes[2].set_title('After PReLU (sparsified)')
axes[2].set_xlabel('Time (ms)'); axes[2].set_ylabel('Filter #')
plt.tight_layout(); plt.show()

before = (feat > 0.01).mean() * 100
after = (enc_active > 0.01).mean() * 100
print(f'Active elements (>0.01): Before PReLU={before:.1f}%, After PReLU={after:.1f}%')


## 6. 双分支注意力 - 局部 vs 全局

In [ ]:
b0_local = activations['block0_local_cemhsa']
b0_global = activations['block0_global_cemhsa']
b9_local = activations['block9_local_cemhsa']
b9_global = activations['block9_global_cemhsa']

print(f'Block 0 Local:  {list(b0_local.shape)}')
print(f'Block 0 Global: {list(b0_global.shape)}')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

if len(b0_local.shape) >= 3:
    lf = b0_local[0, :, :min(50, b0_local.shape[-1])].numpy()
    axes[0,0].imshow(lf, aspect='auto', cmap='RdBu_r')
    axes[0,0].set_title('Block 0 Local Branch (within-chunk)')
    axes[0,0].set_xlabel('Time (within chunk)'); axes[0,0].set_ylabel('Feature dim')

if len(b0_global.shape) >= 3:
    gf = b0_global[0, :, :min(50, b0_global.shape[-1])].numpy()
    axes[0,1].imshow(gf, aspect='auto', cmap='RdBu_r')
    axes[0,1].set_title('Block 0 Global Branch (cross-chunk)')
    axes[0,1].set_xlabel('Chunk index'); axes[0,1].set_ylabel('Feature dim')

if len(b9_local.shape) >= 3:
    lf9 = b9_local[0, :, :min(50, b9_local.shape[-1])].numpy()
    axes[1,0].imshow(lf9, aspect='auto', cmap='RdBu_r')
    axes[1,0].set_title('Block 9 Local Branch (deeper)')

if len(b9_global.shape) >= 3:
    gf9 = b9_global[0, :, :min(50, b9_global.shape[-1])].numpy()
    axes[1,1].imshow(gf9, aspect='auto', cmap='RdBu_r')
    axes[1,1].set_title('Block 9 Global Branch (deeper)')

plt.tight_layout(); plt.show()
print('Block 0 -> shallow, preserves fine details')
print('Block 9 -> deep, captures abstract patterns')


## 7. 掩码融合权重 + 噪声估计

In [ ]:
fw = ckpt.get('mask_fusion_weights')
if fw is not None:
    fw = fw.cpu().numpy() if isinstance(fw, torch.Tensor) else np.array(fw)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(range(1, 11), fw, color=['#1A5276']+['#1C7293']*8+['#2E86C1'], edgecolor='black', linewidth=0.5)
    for i, w in enumerate(fw):
        ax.text(i+1, w, f'{w:.2f}', ha='center', va='bottom', fontsize=10)
    ax.set_xlabel('Layer'); ax.set_ylabel('Weight')
    ax.set_title('Mask Fusion Weights (Softmax)')
    ax.set_xticks(range(1, 11)); ax.grid(axis='y', alpha=0.3)
    plt.show()

dec_out = activations['decoder_out']
if isinstance(dec_out, torch.Tensor):
    fig, axes = plt.subplots(2, 1, figsize=(14, 6))
    dn = dec_out.squeeze().numpy()[:L]
    t = np.arange(len(dn)) / sr * 1000
    axes[0].plot(t, dn, color='#E67E22', linewidth=0.4)
    axes[0].set_title('Decoder Output: Estimated Noise n(t)')
    axes[0].set_xlim(0, 500)
    axes[1].plot(t, enhanced_np, color='#1C7293', linewidth=0.4)
    axes[1].set_title('Enhanced Signal s(t) = x(t) - n(t)')
    axes[1].set_xlim(0, 500)
    plt.tight_layout(); plt.show()


## 8. 频域对比

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

f_c, p_c = welch(clean, sr, nperseg=1024)
f_n, p_n = welch(noisy, sr, nperseg=1024)
f_e, p_e = welch(enhanced_np, sr, nperseg=1024)
mask = (f_c >= 0) & (f_c <= 4000)

axes[0,0].plot(f_c[mask]/1000, 10*np.log10(p_c[mask]), 'k-', lw=1.2, label='Clean')
axes[0,0].plot(f_n[mask]/1000, 10*np.log10(p_n[mask]), color='#E74C3C', lw=0.6, alpha=0.6, label='Noisy')
axes[0,0].plot(f_e[mask]/1000, 10*np.log10(p_e[mask]), '--', color='#1C7293', lw=1.2, label='DCAMF-Net')
axes[0,0].set_title('PSD Comparison'); axes[0,0].legend(fontsize=8)
axes[0,0].set_xlabel('kHz'); axes[0,0].grid(True, alpha=0.3)

f_s, t_s, Sxx = spectrogram(noisy, sr, nperseg=256, noverlap=128)
axes[0,1].pcolormesh(t_s, f_s, 10*np.log10(Sxx+1e-10), shading='auto', cmap='inferno')
axes[0,1].set_title('Noisy Spectrogram'); axes[0,1].set_ylim(0, 4000)

f_se, t_se, Sxx_e = spectrogram(enhanced_np, sr, nperseg=256, noverlap=128)
axes[1,0].pcolormesh(t_se, f_se, 10*np.log10(Sxx_e+1e-10), shading='auto', cmap='inferno')
axes[1,0].set_title('Enhanced Spectrogram'); axes[1,0].set_ylim(0, 4000)

f_sc, t_sc, Sxx_c = spectrogram(clean, sr, nperseg=256, noverlap=128)
axes[1,1].pcolormesh(t_sc, f_sc, 10*np.log10(Sxx_c+1e-10), shading='auto', cmap='inferno')
axes[1,1].set_title('Clean Spectrogram'); axes[1,1].set_ylim(0, 4000)
plt.tight_layout(); plt.show()


## 9. 学习到的滤波器组

In [ ]:
enc_weights = model.encoder.conv.weight.detach().cpu()
filters = enc_weights.squeeze(1).numpy()

fig, axes = plt.subplots(5, 5, figsize=(14, 10))
for i, ax in enumerate(axes.flat):
    if i < 25:
        ax.plot(np.arange(80)/sr*1000, filters[i], linewidth=0.6, color='#1A5276')
        ax.set_title(f'Filter {i}', fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])
    else:
        ax.axis('off')
plt.suptitle('Learned Convolutional Filters (First 25 of 256)', fontsize=14)
plt.tight_layout(); plt.show()

from numpy.fft import fft, fftfreq
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for i, fi in enumerate([0, 10, 50, 128]):
    ax = axes.flat[i]
    f_resp = np.abs(fft(filters[fi], n=1024))
    freq = fftfreq(1024, 1/sr)
    ax.plot(freq[:512], f_resp[:512], color='#1C7293', linewidth=0.8)
    ax.set_title(f'Filter {fi} Frequency Response')
    ax.set_xlabel('Hz'); ax.set_xlim(0, 4000); ax.grid(True, alpha=0.3)
plt.suptitle('Analysis Filter Bank: Frequency Responses', fontsize=14)
plt.tight_layout(); plt.show()


## 10. 残差重建: x(t) - n(t)

In [ ]:
dec_out = activations['decoder_out']
if isinstance(dec_out, torch.Tensor):
    n_est = dec_out.squeeze().numpy()[:L]
    fig, axes = plt.subplots(3, 1, figsize=(14, 8))
    t = np.arange(L) / sr * 1000
    axes[0].plot(t, noisy, color='#E74C3C', linewidth=0.4, label='Noisy x(t)')
    axes[0].plot(t, clean, color='black', linewidth=0.6, alpha=0.7, label='Clean s(t)')
    axes[0].set_title('Input: Noisy + Clean Reference')
    axes[0].legend(fontsize=8); axes[0].set_xlim(0, 300)
    axes[1].fill_between(t, 0, n_est, color='#E67E22', alpha=0.4)
    axes[1].plot(t, n_est, color='#E67E22', linewidth=0.4)
    axes[1].set_title('Estimated Noise n(t)')
    axes[1].set_xlim(0, 300)
    axes[2].plot(t, clean, color='black', linewidth=0.6, alpha=0.7, label='Clean')
    axes[2].plot(t, enhanced_np, color='#1C7293', linewidth=0.5, label='Enhanced')
    axes[2].set_title('Result: Enhanced vs Clean')
    axes[2].legend(fontsize=8); axes[2].set_xlim(0, 300)
    plt.tight_layout(); plt.show()

    print(f'Noise power removed: {np.mean(n_est**2):.4f} ({np.mean(n_est**2)/np.mean(noisy**2)*100:.1f}% of input)')
    print(f'Clean signal power: {np.mean(clean**2):.4f}')
    print(f'Enhanced signal power: {np.mean(enhanced_np**2):.4f}')
